In [ ]:
from datetime import date, datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# from sklearn.preprocessing import OneHotEncoder

1.データ読み込み

In [ ]:
# 使用するカラムを指定

usecols = ["Year", "Month", "Day", "Hour", "Minute", "Temperature", "Clearsky GHI", "Cloud Type", "Dew Point", "DHI", "DNI",
            "GHI", "Relative Humidity","Pressure", "Precipitable Water", "Wind Direction", "Wind Speed"]

In [ ]:
# ファイルの読み込み＋複数年を合算


df_weather_yamanashi = pd.DataFrame()

for year in range(2016, 2021):

    file_path = f"../data/weather_yamanashi_{year}.csv"
    print(file_path)
    df = pd.read_csv(file_path, skiprows=2, usecols=usecols)

    if df_weather_yamanashi.empty:
        # print("empty")
        df_weather_yamanashi = df
    else:
        # print("not empty")
        df_weather_yamanashi =  pd.concat([df_weather_yamanashi, df])

df_weather_yamanashi.reset_index(drop = True)
    


In [ ]:
df_weather_yamanashi.head()

2.データ品質の確認（基本統計量、欠損値、表記ブレの確認）

In [ ]:
# 欠損値の確認

df_weather_yamanashi.isnull().sum()

In [ ]:
# 質的変数の表記ブレの確認
# df_weather_yamanashi["Cloud Type"].unique()
df_weather_yamanashi["Cloud Type"].value_counts()


In [ ]:
# 基本統計量の確認
df_weather_yamanashi.describe()

3.可視化

In [ ]:
#分布の確認

In [ ]:
# GHIのヒストグラム

x = df_weather_yamanashi["GHI"]

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(x, bins = 50)
ax.set_title("GHI")
ax.set_xlabel("x")
ax.set_ylabel('freq')
fig.show()

In [ ]:
# Temparatureのヒストグラム

x = df_weather_yamanashi["Temperature"]

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(x, bins = 50)
ax.set_title("Temperature")
ax.set_xlabel("x")
ax.set_ylabel('freq')
fig.show()

In [ ]:
# 時刻をつくる
# 元データがtz = 0なので、utc指定。
df_weather_yamanashi["dt"] = pd.to_datetime(df_weather_yamanashi[["Year", "Month", "Day", "Hour"]], utc = True)

In [ ]:
# time zoneの処理
# 元データはtz=0, local tz=9なので、hourを+9する。
# 23時とかは32時間とかになるからちゃんと計算する。

df_weather_yamanashi["jst"] = df_weather_yamanashi["dt"] + pd.Timedelta(hours=9)

In [ ]:
#日射量から、発電量を作成
# 発電量 = GHI[W/㎡] ＊ 1 [hour] * 発電効率　＊　温度補正 * パネル面積
# 発電量[Wh/m2] = GHI ＊ 1 ＊ 0.18 * (1 - 0.004 * (Temperature - 25)) * 1
# 本来は外気気温ではなく、パネル温度で補正をかける

# [Wh]
df_weather_yamanashi["pv"] = df_weather_yamanashi["GHI"] * 1 * 0.18 * (1 - 0.004 * (df_weather_yamanashi["Temperature"] - 25))  * 1



In [ ]:
# 時系列で発電量をみる。
# 年毎に分ける

for i in range(2016, 2021):
    x = df_weather_yamanashi["jst"][df_weather_yamanashi["Year"] == i]
    y = df_weather_yamanashi["pv"][df_weather_yamanashi["Year"] == i]

    plt.plot(x, y)
    plt.xlabel("time")
    plt.ylabel("pv")
    plt.show()


In [ ]:
# カクツキはあるが、季節によって発電量も変化している。春〜夏が多くて、秋〜冬が少ない
# 季節性がある
# 年毎の差はほとんどない

In [ ]:
# pvを時間ごとに分解して描画してみる

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(0, 24):
    x = df_weather_yamanashi["jst"][(df_weather_yamanashi["jst"].dt.hour == i) & (df_weather_yamanashi["Year"] == 2016)]
    y = df_weather_yamanashi["pv"][(df_weather_yamanashi["jst"].dt.hour == i) & (df_weather_yamanashi["Year"] == 2016)]

    ax.plot(x, y, label=f"{i}:00")

plt.xlabel("time")
plt.ylabel("pv")
ax.legend()
plt.show()

In [ ]:
# グラフが見えにくくなっているが、発電量に周期性がある

In [ ]:
# pvを日ごとに平均とってみる

# series_pv_mean = df_weather_yamanashi.groupby(df_weather_yamanashi["jst"].dt.dayofyear)["pv"].mean()

series_pv_mean = df_weather_yamanashi.groupby([df_weather_yamanashi["jst"].dt.date])["pv"].mean()

# series_pv_mean = df_weather_yamanashi.groupby([df_weather_yamanashi["jst"].dt.strftime("%m-%d")])["pv"].mean()

plt.plot(series_pv_mean)
# plt.plot(df_pv_mean_by_date.iloc[:, 0], df_pv_mean_by_date.iloc[:, 1])


# plt.tight_layout()
plt.xlabel("time")
plt.ylabel("pv_mean[Wh]")
plt.show()

In [ ]:
# pvを時間ごとに平均とってみる
series_pv_mean = df_weather_yamanashi.groupby(df_weather_yamanashi["jst"].dt.hour)["pv"].mean()

plt.plot(series_pv_mean)
plt.xlabel("hour")
plt.ylabel("pv_mean[Wh]")
plt.show()

In [ ]:
# 時間ごとに、発電量の平均をとると、周期性が顕著

In [ ]:
# pvを気温ごとに平均とってみる
series_pv_mean = df_weather_yamanashi.groupby(df_weather_yamanashi["Temperature"])["pv"].mean()

plt.plot(series_pv_mean)
plt.xlabel("Temperature[℃]")
plt.ylabel("pv_mean[Wh]")
plt.show()

In [ ]:
# 気温が上がれば発電量も増えているのは、気温が高い＝基本晴れている＝日射量が多い、という可能性がたかい。
# 気温の10度〜２０度後半がのばらつきが小さい。曇りなどの気象条件？

In [ ]:
# Cloud typeの分布
# 'Clear': 0, 'Probably Clear': 1, 'Fog': 2, 'Water': 3, 'Super-Cooled Water': 4, 'Mixed': 5, 'Opaque Ice': 6, 'Cirrus': 7, 'Overlapping': 8, 'Overshooting': 9,
df_weather_yamanashi["Cloud Type"].value_counts().plot.bar()
plt.ylabel("freq")
plt.show()

In [ ]:
# 偏りが大きい

In [ ]:
# cloud tyoeごとの発電量の平均
series_pv_mean = df_weather_yamanashi.groupby(df_weather_yamanashi["Cloud Type"])["pv"].mean().plot.bar()
plt.xlabel("Cloud type")
plt.ylabel("pv_mean")
plt.show()



In [ ]:
# cloudtypeごとに発電量に偏りがある

In [ ]:
# cloudtypeごとの発電量の箱ヒゲグラフ


list_pv = []
list_cloudtype = []

for i in df_weather_yamanashi["Cloud Type"].unique():
    print(i)

    list = df_weather_yamanashi["pv"][df_weather_yamanashi["Cloud Type"]==i]

    list_cloudtype.append(i)
    list_pv.append(list)

plt.boxplot(list_pv, labels= list_cloudtype)
plt.xlabel("Cloud type")
plt.ylabel("pv")
plt.show()


In [ ]:
# cloud typeによってはばらつきがおおきい